In [1]:
#!/usr/bin/env python3
"""
DECODING VERIFICATION — QPA + EBA + Full Statistical Analysis
==============================================================
This notebook is the primary analysis notebook for the paper.
Run it on full-mode (32×32) quantum simulation outputs only.

Steps:
  1. Load shares from neqr_shares_eba/set_<label>/
  2. QPA + EBA decode-side verification (quantum circuit)
  3. Secret recovery (quantum decoding circuit)
  4. Full statistical analysis:
       PSNR, MSE, SSIM (per secret image)
       Entropy (secrets vs shares)
       Pixel correlation coefficients (H / V / Diagonal)
       Histograms
  5. QPA tamper detection demo (untampered vs 30% tampered)
  6. EBA eavesdropping detection demo
  7. Multi-image summary table

All numbers in this notebook feed directly into the paper results section.
"""


'\nDECODING VERIFICATION — QPA + EBA + Full Statistical Analysis\n==============================================================\nThis notebook is the primary analysis notebook for the paper.\nRun it on full-mode (32×32) quantum simulation outputs only.\n\nSteps:\n  1. Load shares from neqr_shares_eba/set_<label>/\n  2. QPA + EBA decode-side verification (quantum circuit)\n  3. Secret recovery (quantum decoding circuit)\n  4. Full statistical analysis:\n       PSNR, MSE, SSIM (per secret image)\n       Entropy (secrets vs shares)\n       Pixel correlation coefficients (H / V / Diagonal)\n       Histograms\n  5. QPA tamper detection demo (untampered vs 30% tampered)\n  6. EBA eavesdropping detection demo\n  7. Multi-image summary table\n\nAll numbers in this notebook feed directly into the paper results section.\n'

In [2]:
import os, sys, json, math, time
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import pandas as pd

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from mqt.ddsim import DDSIMProvider

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from qvss_utils import (
    to_bits, classical_parity, image_entropy, calc_metrics,
    correlation_coefficient, get_npos,
    build_share_qpa_eba_circuit, parse_qpa_eba_counts,
    build_decoding_circuit, parse_recovery,
    classical_decode, extract_secret_nibble,
    compare_parity_maps, compare_eba_maps, eba_accuracy,
    apply_tampering,
)

# ═══════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════
SHARE_DIR = 'neqr_shares_eba'   # canonical output dir from encoding
OUT_DIR   = 'neqr_recovered_bigimages'
ROWS      = 32
COLS      = 32
BITS      = 8
SHOTS     = 8192

# Image set labels to analyse (must match encoding labels)
IMAGE_SETS = ['A']   # add more labels e.g. ['A', 'B', 'C'] for multi-image

provider    = DDSIMProvider()
SIM_BACKEND = provider.get_backend('qasm_simulator')
os.makedirs(OUT_DIR, exist_ok=True)
print(f"[Config]  SHARE_DIR={SHARE_DIR}  ROWS={ROWS}  COLS={COLS}  SHOTS={SHOTS}")


[Config]  SHARE_DIR=neqr_shares_eba  ROWS=32  COLS=32  SHOTS=8192


In [3]:
# ── Loading helpers ───────────────────────────────────────────────────
def load_set(label):
    """Load shares, parity maps, eba_maps and originals for one image set."""
    set_dir = os.path.join(SHARE_DIR, f'set_{label}')
    print(f"\n[Loading set '{label}' from {set_dir}]")

    shares, parity_maps, eba_maps = {}, {}, {}
    for name in ('S1', 'S2', 'S3'):
        p = os.path.join(set_dir, f'{name}.npy')
        shares[name] = np.load(p)
        print(f"  {name}: {shares[name].shape}  "
              f"min={shares[name].min()}  max={shares[name].max()}")

        pp = os.path.join(set_dir, f'parity_map_{name}.npy')
        parity_maps[name] = np.load(pp) if os.path.exists(pp) else             -np.ones((ROWS, COLS), int)

        ep = os.path.join(set_dir, f'eba_map_{name}.npy')
        eba_maps[name] = np.load(ep).astype(int) if os.path.exists(ep) else             np.zeros((ROWS, COLS), int)

    originals = {}
    for i in range(1, 3):
        p = os.path.join(set_dir, f'G{i}_original.png')
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            originals[f'G{i}'] = img
            print(f"  G{i} original loaded: {img.shape}")

    meta = {}
    mp = os.path.join(set_dir, 'encoding_meta.json')
    if os.path.exists(mp):
        with open(mp) as f: meta = json.load(f)
        print(f"  Encoding meta loaded: qpa_ok={meta.get('qpa_ok')}, "
              f"eba_ok={meta.get('eba_ok')}")

    return shares, parity_maps, eba_maps, originals, meta


def print_auth_result(qpa_stats, eba_stats, label=''):
    """Print QPA and EBA accuracy table."""
    print(f"  ── Auth [{label}] ──────────────────────────")
    print(f"  {'Share':<6} {'QPA correct':<16} {'QPA acc%':<12} "
          f"{'EBA correct':<16} {'EBA acc%':<12} Status")
    for s in ['S1','S2','S3']:
        qt = qpa_stats[s]['correct'] + qpa_stats[s]['wrong']
        qa = 100*qpa_stats[s]['correct']/max(1,qt)
        et = eba_stats[s]['correct'] + eba_stats[s]['wrong']
        ea = 100*eba_stats[s]['correct']/max(1,et)
        ok = '✓ PASS' if qa >= 99 and ea >= 99 else '✗ FAIL'
        print(f"  {s:<6} {qpa_stats[s]['correct']}/{qt:<12} {qa:<12.2f} "
              f"{eba_stats[s]['correct']}/{et:<12} {ea:<12.2f} {ok}")


In [4]:
# ── Quantum decode-side verification ─────────────────────────────────
def run_decode_verification(shares, enc_parity, enc_eba_maps, label=''):
    """
    Run build_share_qpa_eba_circuit on received shares.
    Checks:
      1. QPA accuracy  (Bell parity consistent with share values)
      2. EBA accuracy  (Bell correlations intact = no eavesdropping)
      3. Compare parity maps  (encoding vs decoding = tamper detection)
      4. Compare EBA MSB maps (encoding vs decoding = MSB tamper detection)
    """
    print(f"\n[Decode-side QPA+EBA verification — {label}]")
    t0 = time.time()
    qc      = build_share_qpa_eba_circuit(shares, ROWS, COLS)
    counts  = SIM_BACKEND.run(qc, shots=SHOTS).result().get_counts()
    elapsed = time.time() - t0
    print(f"  Circuit: {qc.num_qubits} qubits  depth={qc.depth()}  time={elapsed:.1f}s")

    dec_shares, dec_parity, qpa_stats, eba_stats =         parse_qpa_eba_counts(counts, ROWS, COLS)

    print_auth_result(qpa_stats, eba_stats, label)

    print("\n  QPA tamper comparison (enc parity vs dec parity):")
    qpa_cmp = compare_parity_maps(enc_parity, dec_parity, ROWS, COLS, label=label)

    print("\n  EBA MSB comparison (enc eba_map vs decoded share MSBs):")
    eba_cmp = compare_eba_maps(enc_eba_maps, dec_shares, ROWS, COLS, label=label)

    ea = eba_accuracy(eba_stats)
    print(f"\n  EBA eavesdrop check: "
          f"S1={ea['S1']:.2f}%  S2={ea['S2']:.2f}%  S3={ea['S3']:.2f}%  "
          f"(~100%=clean, ~50%=eavesdropped)")

    return dec_shares, dec_parity, qpa_stats, eba_stats, qpa_cmp, eba_cmp


def run_secret_recovery(shares, label=''):
    """Run build_decoding_circuit to recover R1 and R2."""
    print(f"\n[Secret recovery — {label}]")
    t0 = time.time()
    qc      = build_decoding_circuit(shares, ROWS, COLS)
    counts  = SIM_BACKEND.run(qc, shots=SHOTS).result().get_counts()
    elapsed = time.time() - t0
    rec = parse_recovery(counts, ROWS, COLS)
    R1, R2  = rec['R1'], rec['R2']
    G1_disp, G2_disp = extract_secret_nibble(R1, R2)
    print(f"  Done in {elapsed:.1f}s")
    print(f"  R1: range=[{R1.min()},{R1.max()}]  "
          f"G1_disp: range=[{G1_disp.min()},{G1_disp.max()}]")
    print(f"  R2: range=[{R2.min()},{R2.max()}]  "
          f"G2_disp: range=[{G2_disp.min()},{G2_disp.max()}]")
    return R1, R2, G1_disp, G2_disp


In [5]:
# ── Statistical analysis ──────────────────────────────────────────────
def statistical_analysis(originals, shares, G1_disp, G2_disp, label=''):
    """
    Full statistical analysis for the paper.
    Returns metrics_df (DataFrame) and correlation_df (DataFrame).
    """
    print(f"\n[Statistical analysis — {label}]")
    print("=" * 70)

    # ── PSNR / SSIM / MSE ───────────────────────────────────────────
    metrics_rows = []
    G1_orig = originals.get('G1')
    G2_orig = originals.get('G2')

    if G1_orig is not None:
        # Compare G1 MSB nibble (4 high bits) — what Code 1 actually recovers
        G1_msb_orig = ((G1_orig >> 4) << 4).astype(np.uint8)
        m = calc_metrics(G1_msb_orig, G1_disp)
        m['image'] = 'G1 (MSB nibble)'; m['label'] = label
        metrics_rows.append(m)
        psnr_str = '+∞' if m['PSNR'] == float('inf') else f"{m['PSNR']:.4f}"
        print(f"  G1: PSNR={psnr_str} dB  SSIM={m['SSIM']:.6f}  MSE={m['MSE']:.4f}  "
              f"px_mismatch={m['px_mismatch']}")

    if G2_orig is not None:
        G2_msb_orig = ((G2_orig >> 4) << 4).astype(np.uint8)
        m = calc_metrics(G2_msb_orig, G2_disp)
        m['image'] = 'G2 (MSB nibble)'; m['label'] = label
        metrics_rows.append(m)
        psnr_str = '+∞' if m['PSNR'] == float('inf') else f"{m['PSNR']:.4f}"
        print(f"  G2: PSNR={psnr_str} dB  SSIM={m['SSIM']:.6f}  MSE={m['MSE']:.4f}  "
              f"px_mismatch={m['px_mismatch']}")

    # ── Entropy ──────────────────────────────────────────────────────
    print("\n  Entropy (bits):")
    imgs_for_entropy = {}
    if G1_orig is not None: imgs_for_entropy['G1 (secret)'] = G1_orig
    if G2_orig is not None: imgs_for_entropy['G2 (secret)'] = G2_orig
    imgs_for_entropy.update({f'S{i+1} (share)': shares[f'S{i+1}'] for i in range(3)})
    imgs_for_entropy['G1 (recovered)'] = G1_disp
    imgs_for_entropy['G2 (recovered)'] = G2_disp

    entropy_rows = []
    for name, img in imgs_for_entropy.items():
        ent = image_entropy(img)
        entropy_rows.append({'image': name, 'entropy': ent, 'label': label})
        print(f"    {name:<22}: {ent:.4f} bits")

    # ── Pixel correlation ─────────────────────────────────────────────
    print("\n  Pixel correlation coefficients:")
    corr_rows = []
    for cname, cimg in [('G1', G1_orig), ('G2', G2_orig),
                         ('S1', shares['S1']), ('S2', shares['S2']), ('S3', shares['S3']),
                         ('G1_rec', G1_disp), ('G2_rec', G2_disp)]:
        if cimg is None: continue
        row = {'image': cname, 'label': label}
        for d in ['horizontal', 'vertical', 'diagonal']:
            cc = correlation_coefficient(cimg, d)
            row[d] = cc
        corr_rows.append(row)
        print(f"    {cname:<10}: H={row['horizontal']:+.4f}  "
              f"V={row['vertical']:+.4f}  D={row['diagonal']:+.4f}")

    metrics_df = pd.DataFrame(metrics_rows) if metrics_rows else pd.DataFrame()
    entropy_df = pd.DataFrame(entropy_rows)
    corr_df    = pd.DataFrame(corr_rows)

    return metrics_df, entropy_df, corr_df


In [6]:
# ── Plot functions ────────────────────────────────────────────────────
def upscale(img, factor=8):
    from PIL import Image
    return np.array(Image.fromarray(img.astype(np.uint8))
                    .resize((img.shape[1]*factor, img.shape[0]*factor),
                            Image.NEAREST))

def plot_visual_pipeline(originals, shares, G1_disp, G2_disp,
                         label='', out_dir=OUT_DIR):
    """Standard paper figure: secrets → shares → recovered."""
    DISP = 200
    BG   = 'white'; TEXT = 'black'
    fig  = plt.figure(figsize=(14, 11), facecolor=BG)
    gs   = gridspec.GridSpec(3, 6, figure=fig,
                             hspace=0.42, wspace=0.06,
                             left=0.02, right=0.98, top=0.96, bottom=0.07)

    def place(row, col, span, img, xlabel, subtitle='', border=False):
        ax = fig.add_subplot(gs[row, col:col+span])
        ax.imshow(upscale(img, DISP//img.shape[0]), cmap='gray', vmin=0, vmax=255)
        ax.set_xlabel(xlabel, color=TEXT, fontsize=11, fontstyle='italic', labelpad=4)
        ax.xaxis.set_label_position('bottom')
        ax.axis('off')
        if subtitle:
            ax.set_title(subtitle, color='#444', fontsize=8.5, pad=3)
        if border:
            for sp in ax.spines.values():
                sp.set_visible(True); sp.set_edgecolor('black'); sp.set_linewidth(1)
        return ax

    G1 = originals.get('G1')
    G2 = originals.get('G2')
    if G1 is None: G1 = np.zeros((ROWS, COLS), np.uint8)
    if G2 is None: G2 = np.zeros((ROWS, COLS), np.uint8)

    place(0, 0, 2, shares['S1'], '(a) Cover → $S_1$',
          f'Entropy={image_entropy(shares["S1"]):.3f} bits')
    place(0, 2, 2, G1, '(b) Secret $G_1$',
          f'Entropy={image_entropy(G1):.3f} bits', border=True)
    place(0, 4, 2, G2, '(c) Secret $G_2$',
          f'Entropy={image_entropy(G2):.3f} bits', border=True)

    place(1, 0, 2, shares['S1'], '(d) Share $S_1$',
          f'Entropy={image_entropy(shares["S1"]):.3f} bits')
    place(1, 2, 2, shares['S2'], '(e) Share $S_2$',
          f'Entropy={image_entropy(shares["S2"]):.3f} bits')
    place(1, 4, 2, shares['S3'], '(f) Share $S_3$',
          f'Entropy={image_entropy(shares["S3"]):.3f} bits')

    G1_msb = ((G1>>4)<<4).astype(np.uint8)
    G2_msb = ((G2>>4)<<4).astype(np.uint8)
    m1 = calc_metrics(G1_msb, G1_disp)
    m2 = calc_metrics(G2_msb, G2_disp)
    p1 = '+∞' if m1['PSNR']==float('inf') else f"{m1['PSNR']:.2f}"
    p2 = '+∞' if m2['PSNR']==float('inf') else f"{m2['PSNR']:.2f}"

    place(2, 1, 2, G1_disp, '(g) Recovered $R_1$',
          f'PSNR={p1} dB  SSIM={m1["SSIM"]:.4f}', border=True)
    place(2, 3, 2, G2_disp, '(h) Recovered $R_2$',
          f'PSNR={p2} dB  SSIM={m2["SSIM"]:.4f}', border=True)

    fig.text(0.5, 0.025,
        '(a)–(c) Input images; (d)–(f) Share images (noise-like, high entropy); '
        '(g)–(h) Losslessly recovered secrets via Code 1 reverse logic. '
        'Images 32×32 displayed 8× (nearest-neighbour).',
        ha='center', fontsize=9, color='black', style='italic')

    out = os.path.join(out_dir, f'visual_pipeline_{label}.png')
    plt.savefig(out, dpi=220, bbox_inches='tight', facecolor=BG)
    plt.close(); print(f"  Visual pipeline saved → {out}")


def plot_histograms(originals, shares, G1_disp, G2_disp,
                    label='', out_dir=OUT_DIR):
    """Pixel distribution comparison for paper security analysis."""
    BG='#161b22'; TEXT='#e6edf3'; BORDER='#30363d'
    GOLD='#ffd600'; BLUE='#4fc3f7'
    G1 = originals.get('G1', np.zeros((ROWS,COLS),np.uint8))
    G2 = originals.get('G2', np.zeros((ROWS,COLS),np.uint8))
    imgs  = [G1, G2, shares['S1'], shares['S2'], shares['S3']]
    names = ['Secret $G_1$', 'Secret $G_2$', 'Share $S_1$', 'Share $S_2$', 'Share $S_3$']
    hcols = [GOLD, GOLD, BLUE, BLUE, BLUE]

    fig, axes = plt.subplots(2, 5, figsize=(22, 8))
    fig.patch.set_facecolor(BG)

    for col, (img, name, hc) in enumerate(zip(imgs, names, hcols)):
        ai = axes[0, col]
        ai.imshow(upscale(img), cmap='gray', vmin=0, vmax=255)
        border_col = GOLD if col < 2 else BLUE
        for sp in ai.spines.values():
            sp.set_visible(True); sp.set_edgecolor(border_col); sp.set_linewidth(2)
        ai.set_title(name, color=border_col, fontsize=10, fontweight='bold')
        ai.axis('off')

        ah = axes[1, col]
        ah.set_facecolor('#161b22')
        ah.tick_params(colors=TEXT, labelsize=8)
        for sp in ah.spines.values(): sp.set_color(BORDER)
        ah.spines['top'].set_visible(False); ah.spines['right'].set_visible(False)
        hist, bins = np.histogram(img, bins=32, range=(0,255))
        ah.bar(bins[:-1], hist, width=8, color=hc, alpha=0.85)
        ent = image_entropy(img)
        ah.set_title(f'H={ent:.3f} bits', color=TEXT, fontsize=10, fontweight='bold')
        ah.set_xlabel('Pixel value', color=TEXT, fontsize=8)
        ah.grid(color=BORDER, linestyle='--', alpha=0.4)

    fig.suptitle(f'Pixel Distribution — Secrets vs Shares [{label}]\n'
                 f'High share entropy confirms uniform distribution (security proof)',
                 color=TEXT, fontsize=12, fontweight='bold')
    plt.tight_layout(pad=1.0)
    out = os.path.join(out_dir, f'histograms_{label}.png')
    plt.savefig(out, dpi=180, bbox_inches='tight', facecolor=BG)
    plt.close(); print(f"  Histogram plot saved → {out}")


def plot_qpa_eba_tamper(shares, enc_parity, enc_eba_maps,
                        label='', out_dir=OUT_DIR):
    """
    QPA tamper localization + EBA MSB change map.
    Compares encoding-time maps vs decode-time on: clean and 30%-tampered S1.
    """
    tc = LinearSegmentedColormap.from_list('tc', ['#0d1117', '#ff1744'])
    BG='#161b22'; TEXT='#e6edf3'

    def tamper_30(sh):
        t = {k: v.copy() for k,v in sh.items()}
        np.random.seed(42)
        mask = np.random.random((ROWS, COLS)) < 0.30
        t['S1'][mask] ^= 0xFF
        return t, mask

    t_shares, t_mask = tamper_30(shares)

    def qpa_map(sh):
        pm = {}
        for s in ['S1','S2','S3']:
            pm[s] = np.vectorize(classical_parity)(sh[s]).astype(int)
        return pm

    orig_parity  = qpa_map(shares)
    tamp_parity  = qpa_map(t_shares)
    orig_eba     = {k: ((shares[k]>>7)&1).astype(int)   for k in ['S1','S2','S3']}
    tamp_eba_msb = {k: ((t_shares[k]>>7)&1).astype(int) for k in ['S1','S2','S3']}

    fig, axes = plt.subplots(4, 3, figsize=(14, 18))
    fig.patch.set_facecolor('#0d1117')

    row_titles = ['Untampered share', '30% tampered S1',
                  'QPA detection map', 'EBA MSB change map']
    share_names = ['S1', 'S2', 'S3']

    for col, sname in enumerate(share_names):
        # Row 0: untampered share
        axes[0,col].imshow(upscale(shares[sname]), cmap='gray', vmin=0, vmax=255)
        axes[0,col].set_title(f'{sname} (clean)', color=TEXT, fontsize=10, fontweight='bold')
        axes[0,col].axis('off')

        # Row 1: tampered share
        axes[1,col].imshow(upscale(t_shares[sname]), cmap='gray', vmin=0, vmax=255)
        status = 'TAMPERED' if sname=='S1' else 'clean'
        axes[1,col].set_title(f'{sname} ({status})',
                               color='#ff1744' if sname=='S1' else TEXT,
                               fontsize=10, fontweight='bold')
        axes[1,col].axis('off')

        # Row 2: QPA detection (encode-time parity vs decode-time parity)
        enc_p   = np.array(enc_parity[sname])
        det_qpa = np.where(enc_p >= 0, (enc_p != tamp_parity[sname]).astype(np.uint8), 0)
        n_det   = int(det_qpa.sum())
        axes[2,col].imshow(upscale(det_qpa*255), cmap=tc, vmin=0, vmax=255)
        axes[2,col].set_title(f'QPA: {n_det} tampered px detected',
                               color='#ff1744' if n_det>0 else '#00e676',
                               fontsize=9, fontweight='bold')
        axes[2,col].axis('off')

        # Row 3: EBA MSB change map (encoding-time MSB vs tampered share MSB)
        det_eba = (enc_eba_maps[sname] != tamp_eba_msb[sname]).astype(np.uint8)
        n_eba   = int(det_eba.sum())
        axes[3,col].imshow(upscale(det_eba*255), cmap=tc, vmin=0, vmax=255)
        axes[3,col].set_title(f'EBA MSB: {n_eba} changes detected',
                               color='#ff1744' if n_eba>0 else '#00e676',
                               fontsize=9, fontweight='bold')
        axes[3,col].axis('off')

    for ax in axes.flat: ax.set_facecolor('#161b22')
    for row_idx, rtitle in enumerate(row_titles):
        axes[row_idx, 0].set_ylabel(rtitle, color='#ffd600',
                                     fontsize=10, fontweight='bold', labelpad=8)

    fig.suptitle(f'QPA + EBA Tamper & Eavesdrop Detection [{label}]\n'
                 f'30% full-byte tampering applied to S1 only',
                 color=TEXT, fontsize=12, fontweight='bold')
    plt.tight_layout(pad=0.4)
    out = os.path.join(out_dir, f'tamper_detection_{label}.png')
    plt.savefig(out, dpi=180, bbox_inches='tight', facecolor='#0d1117')
    plt.close(); print(f"  Tamper detection plot saved → {out}")


In [7]:
# ═══════════════════════════════════════════════════════════════════
# MAIN EXECUTION
# ═══════════════════════════════════════════════════════════════════
print("=" * 70)
print("QVSS DECODING VERIFICATION & STATISTICAL ANALYSIS")
print("=" * 70)

all_metrics  = []
all_entropy  = []
all_corr     = []

for label in IMAGE_SETS:
    print(f"\n{'='*60}")
    print(f"  IMAGE SET: {label}")
    print(f"{'='*60}")

    shares, parity_maps, eba_maps, originals, enc_meta = load_set(label)

    # Step 1: Decode-side QPA + EBA quantum verification
    dec_shares, dec_parity, qpa_stats, eba_stats, qpa_cmp, eba_cmp =         run_decode_verification(shares, parity_maps, eba_maps, label=f'{label}_clean')

    # Step 2: Quantum secret recovery
    R1, R2, G1_disp, G2_disp = run_secret_recovery(shares, label=label)

    # Step 3: Statistical analysis
    mdf, edf, cdf = statistical_analysis(originals, shares, G1_disp, G2_disp, label=label)
    all_metrics.append(mdf)
    all_entropy.append(edf)
    all_corr.append(cdf)

    # Step 4: Visual figures
    set_out = os.path.join(OUT_DIR, f'set_{label}')
    os.makedirs(set_out, exist_ok=True)
    plot_visual_pipeline(originals, shares, G1_disp, G2_disp, label=label, out_dir=set_out)
    plot_histograms(originals, shares, G1_disp, G2_disp, label=label, out_dir=set_out)
    plot_qpa_eba_tamper(shares, parity_maps, eba_maps, label=label, out_dir=set_out)

    # Save recovered images
    np.save(os.path.join(set_out, 'R1.npy'), R1)
    np.save(os.path.join(set_out, 'R2.npy'), R2)
    cv2.imwrite(os.path.join(set_out, 'G1_recovered.png'), G1_disp)
    cv2.imwrite(os.path.join(set_out, 'G2_recovered.png'), G2_disp)

# ── Multi-image summary ──────────────────────────────────────────────
if len(IMAGE_SETS) > 1 and all_metrics:
    import pandas as pd
    metrics_all = pd.concat(all_metrics, ignore_index=True)
    print("\n" + "=" * 70)
    print("MULTI-IMAGE SUMMARY")
    print("=" * 70)
    summary = metrics_all.groupby('image').agg(
        PSNR_mean=('PSNR', lambda x: x[x != float('inf')].mean()),
        SSIM_mean=('SSIM', 'mean'),
        MSE_mean=('MSE', 'mean'),
        px_mismatch_mean=('px_mismatch', 'mean'),
    ).reset_index()
    print(summary.to_string(index=False))
    summary.to_csv(os.path.join(OUT_DIR, 'metrics_summary.csv'), index=False)

print("\nDONE. Run decoding_verification_robustness.ipynb for tamper sweep.")


QVSS DECODING VERIFICATION & STATISTICAL ANALYSIS

  IMAGE SET: A

[Loading set 'A' from neqr_shares_eba\set_A]
  S1: (32, 32)  min=17  max=238
  S2: (32, 32)  min=27  max=206
  S3: (32, 32)  min=32  max=255
  G1 original loaded: (32, 32)
  G2 original loaded: (32, 32)
  Encoding meta loaded: qpa_ok=True, eba_ok=True

[Decode-side QPA+EBA verification — A_clean]
  Circuit: 43 qubits  depth=35353  time=136.5s
  ── Auth [A_clean] ──────────────────────────
  Share  QPA correct      QPA acc%     EBA correct      EBA acc%     Status
  S1     8192/8192         100.00       8192/8192         100.00       ✓ PASS
  S2     8192/8192         100.00       8192/8192         100.00       ✓ PASS
  S3     8192/8192         100.00       8192/8192         100.00       ✓ PASS

  QPA tamper comparison (enc parity vs dec parity):
  S1 [A_clean]: 1022/1022 match (100.00%)  -> AUTHENTIC
  S2 [A_clean]: 1022/1022 match (100.00%)  -> AUTHENTIC
  S3 [A_clean]: 1022/1022 match (100.00%)  -> AUTHENTIC

  EBA MSB